# PLN de microtextos: ¿qué se cuenta sobre Torrevieja?

Trabajaremos con `datos/torrevieja_tweets_sample.csv`, una muestra de publicaciones breves. El objetivo es **entender un proceso de análisis**, no dar por correctas las predicciones de un modelo.

Al finalizar podrás:
1. Inspeccionar y preparar los datos sin perder sus identificadores.
2. Distinguir la limpieza para contar palabras de la preparación para modelos.
3. Construir y criticar un clasificador léxico de sentimiento.
4. Opcionalmente, usar modelos en español de sentimiento y entidades nombradas (NER).
5. Revisar errores y comunicar las limitaciones de los resultados.

**Tiempo orientativo:** 45–60 minutos para la parte básica; 20–30 adicionales para los modelos. Los mensajes reales pueden contener lenguaje ofensivo. Evita publicar nombres de usuarios o ejemplos identificables en las entregas.

## 0. Clonar el repositorio y preparar el entorno

La primera celda de código clona `https://github.com/chemacecilia/SII-UPV.git` en la ubicación fija `/tmp/SII-UPV`, válida también en Google Colab. Los datos se leerán de `/tmp/SII-UPV/datos/torrevieja_tweets_sample.csv`, independientemente de dónde se haya abierto el notebook.

Se necesita Git y acceso a Internet para la primera clonación. Si ya existe una copia de este repositorio en esa ubicación, se reutiliza sin sobrescribir ni actualizar archivos. `/tmp` es almacenamiento temporal: en una sesión nueva puede ser necesario clonar otra vez.

La parte básica requiere `pandas` y `matplotlib`. Si faltan, ejecuta `%pip install pandas matplotlib` en una celda y reinicia el kernel cuando sea necesario.

In [ ]:
from pathlib import Path
import subprocess

URL_REPOSITORIO = 'https://github.com/chemacecilia/SII-UPV.git'
RUTA_REPOSITORIO = Path('/tmp/SII-UPV')
RUTA_DATOS = RUTA_REPOSITORIO / 'datos' / 'torrevieja_tweets_sample.csv'

if not RUTA_REPOSITORIO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', 'main', URL_REPOSITORIO, str(RUTA_REPOSITORIO)],
        check=True,
    )
else:
    if not (RUTA_REPOSITORIO / '.git').exists():
        raise FileExistsError(f'{RUTA_REPOSITORIO} ya existe y no es el clon esperado; revisa esa carpeta.')
    origen = subprocess.run(
        ['git', '-C', str(RUTA_REPOSITORIO), 'remote', 'get-url', 'origin'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if origen.rstrip('/').removesuffix('.git') != URL_REPOSITORIO.removesuffix('.git'):
        raise ValueError('La carpeta fija pertenece a otro repositorio; revísala antes de continuar.')
    print('Se reutiliza el repositorio existente:', RUTA_REPOSITORIO)

if not RUTA_DATOS.is_file():
    raise FileNotFoundError(f'No se encuentra el fichero de datos esperado: {RUTA_DATOS}')
print('Fichero de datos:', RUTA_DATOS)

In [ ]:
import re
import unicodedata
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEMILLA = 42
MAX_TEXTOS_MODELOS = 30  # Muestra reproducible; usa None para todos (más lento).
EJECUTAR_MODELOS = False  # Cambia a True solo tras instalar las dependencias opcionales.

## 1. Cargar y explorar

El CSV usa comas y codificación UTF-8. Guardamos `id` como texto: un identificador no es una cantidad y puede perder precisión si pasa por un número decimal.

Columnas: `id`, `user`, `fecha`, `idioma`, `texto` y contadores de interacción `retweeted`, `replyed`, `liked`. Estos contadores no son etiquetas de sentimiento. No necesitamos el nombre del usuario para este análisis.

In [ ]:
df_original = pd.read_csv(RUTA_DATOS, encoding='utf-8', dtype={'id': 'string', 'texto': 'string'})
requeridas = {'id', 'fecha', 'idioma', 'texto'}
if not requeridas.issubset(df_original.columns):
    raise ValueError(f'Faltan columnas: {sorted(requeridas - set(df_original.columns))}')
print(f'Filas originales: {len(df_original):,}')
display(df_original[['id', 'fecha', 'idioma', 'texto']].head())
display(df_original[['id', 'fecha', 'idioma', 'texto']].isna().sum().rename('nulos').to_frame())
display(df_original['idioma'].value_counts(dropna=False).rename('mensajes').to_frame())

In [ ]:
df = df_original.copy()
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce', utc=True)
texto_valido = df['texto'].notna() & df['texto'].fillna('').str.strip().ne('')
print('Textos nulos o vacíos:', int((~texto_valido).sum()))
df = df.loc[texto_valido].copy()
df['texto'] = df['texto'].str.strip()
print('IDs repetidos descartados:', int(df['id'].notna().mul(df['id'].duplicated()).sum()))
# Si un ID falta, no agrupamos todos los nulos como si fueran un mismo mensaje.
df = df.loc[df['id'].isna() | ~df['id'].duplicated()].copy()
print('Fechas no interpretables:', int(df['fecha'].isna().sum()))
print('Textos idénticos (se conservan):', int(df['texto'].duplicated().sum()))
print('Intervalo temporal:', df['fecha'].min(), '—', df['fecha'].max())
# La etiqueta de idioma es metadato, no una garantía de que todo el mensaje esté en español.
df = df.loc[df['idioma'].astype('string').str.lower().eq('es').fillna(False)].copy()
if df.empty:
    raise ValueError('No quedan textos en español para analizar.')
df = df.reset_index(drop=True)
print('Mensajes disponibles para el análisis en español:', len(df))

**Para discutir:** ¿un mensaje que contiene “Torrevieja” habla necesariamente de la ciudad? ¿Una muestra tomada durante varios años representa lo que piensan sus habitantes hoy? Examina algunos mensajes antes de responder.

## 2. Dos representaciones para dos tareas

- **Vocabulario:** minúsculas y palabras sin URLs ni menciones; conservamos las palabras de los hashtags. Quitaremos algunas palabras frecuentes al contar.
- **Modelos:** mantenemos mayúsculas, tildes, puntuación, emojis y negaciones. Para sentimiento sustituimos URLs y menciones por marcadores. Para NER usamos el original, pues las mayúsculas y el contexto ayudan a reconocer nombres.

Nunca sobrescribimos `texto`. No eliminamos “no”, “sin” ni “nunca”: “me gusta” y “no me gusta” no expresan lo mismo. Tampoco borramos “Torrevieja” del texto; solo la excluiremos del gráfico de vocabulario por ser el término de búsqueda.

In [ ]:
PATRON_URL = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
PATRON_MENCION = re.compile(r'(?<!\w)@\w+')
PATRON_PALABRA = re.compile(r"[^\W\d_]+", flags=re.UNICODE)


def normalizar_unicode(texto):
    return unicodedata.normalize('NFC', str(texto))


def tokens_vocabulario(texto):
    texto = normalizar_unicode(texto).lower()
    texto = PATRON_URL.sub(' ', texto)
    texto = PATRON_MENCION.sub(' ', texto)
    return PATRON_PALABRA.findall(texto)


def preparar_sentimiento(texto):
    texto = normalizar_unicode(texto)
    texto = PATRON_URL.sub(' URL ', texto)
    texto = PATRON_MENCION.sub(' @usuario ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


df['tokens'] = df['texto'].map(tokens_vocabulario)
df['texto_modelo'] = df['texto'].map(preparar_sentimiento)
display(df[['texto', 'tokens', 'texto_modelo']].head(5))

# Contraste didáctico: la negación y los emojis sobreviven para los modelos.
for ejemplo in ['Me gusta #Torrevieja 😊', 'No me gusta #Torrevieja 😞',
                '@ana ¡Qué playa! https://ejemplo.org']:
    print(ejemplo, '→', preparar_sentimiento(ejemplo))

## 3. ¿Qué palabras aparecen?

La lista siguiente es pequeña y explícita para que puedas modificarla sin descargar recursos. No es un diccionario completo de palabras vacías. Contamos **apariciones**, no personas ni opiniones independientes. “Playa” y “playas” cuentan por separado: todavía no hacemos lematización.

In [ ]:
PALABRAS_FRECUENTES = set("""
a al algo como con de del desde el ella en es esta este esto fue ha hay la las le lo los
me mi mis muy o para pero por que se si su sus te tiene tu un una uno unos unas y ya yo
""".split())
TERMINOS_BUSQUEDA = {'torrevieja'}
excluir = PALABRAS_FRECUENTES | TERMINOS_BUSQUEDA
conteo = Counter(t for tokens in df['tokens'] for t in tokens if len(t) > 2 and t not in excluir)
top_palabras = pd.DataFrame(conteo.most_common(20), columns=['palabra', 'apariciones'])
display(top_palabras)
if not top_palabras.empty:
    top_palabras.set_index('palabra')['apariciones'].sort_values().plot.barh(figsize=(9, 6))
    plt.title('Palabras frecuentes en la muestra (sin el término de búsqueda)')
    plt.xlabel('Número de apariciones')
    plt.tight_layout()
    plt.show()

## 4. Sentimiento: una referencia sencilla que podemos explicar

Antes de usar un modelo, probamos un pequeño léxico: cada término positivo suma 1 y cada negativo resta 1. Una negación invierte el valor de un término si aparece entre las tres palabras anteriores, dentro de la misma frase.

**Es una heurística educativa, no un clasificador validado.** La etiqueta `SIN_EVIDENCIA` indica empate o falta de palabras conocidas; no equivale a “neutral”. No entiende bien ironía, dobles negaciones, emojis, intensidad ni a quién se dirige la opinión. Por ejemplo, “no solo es bonita” puede confundirla. El sentimiento del mensaje tampoco implica una valoración de la ciudad.

In [ ]:
POSITIVAS = {'bueno', 'buena', 'bonito', 'bonita', 'bonitas', 'genial', 'feliz',
             'encanta', 'encantan', 'gusta', 'gustan', 'maravilloso', 'preciosa', 'excelente'}
NEGATIVAS = {'malo', 'mala', 'horrible', 'odio', 'triste', 'sucio', 'sucia',
             'terrible', 'asco', 'miedo', 'vergüenza', 'peor', 'muerto', 'murió'}
NEGACIONES = {'no', 'nunca', 'jamás', 'sin', 'ni'}


def sentimiento_lexico(texto):
    evidencias = []
    puntuacion = 0
    # La ventana no cruza signos que suelen delimitar frases.
    for frase in re.split(r'[.!?;:\n]+', normalizar_unicode(texto)):
        tokens = tokens_vocabulario(frase)
        for i, token in enumerate(tokens):
            valor = int(token in POSITIVAS) - int(token in NEGATIVAS)
            if valor:
                negado = any(t in NEGACIONES for t in tokens[max(0, i - 3):i])
                valor = -valor if negado else valor
                puntuacion += valor
                evidencias.append(f'{token}: {valor:+d}')
    etiqueta = 'POS' if puntuacion > 0 else 'NEG' if puntuacion < 0 else 'SIN_EVIDENCIA'
    return {'sentimiento_lexico': etiqueta, 'puntuacion_lexica': puntuacion,
            'evidencias': ', '.join(evidencias)}


resultados_lexicos = pd.DataFrame(df['texto'].map(sentimiento_lexico).tolist(), index=df.index)
df = df.drop(columns=resultados_lexicos.columns, errors='ignore').join(resultados_lexicos)
ejemplos = pd.DataFrame({'texto': ['Me gusta la playa', 'No me gusta la playa',
                                 'Voy a la playa', 'Genial, otra vez basura en la playa']})
display(ejemplos.join(pd.DataFrame(ejemplos['texto'].map(sentimiento_lexico).tolist())))
display(df[['texto', 'sentimiento_lexico', 'puntuacion_lexica', 'evidencias']].sample(min(8, len(df)), random_state=SEMILLA))

In [ ]:
orden_lexico = ['POS', 'SIN_EVIDENCIA', 'NEG']
resumen_lexico = df['sentimiento_lexico'].value_counts().reindex(orden_lexico, fill_value=0).rename('mensajes').to_frame()
resumen_lexico['porcentaje'] = (100 * resumen_lexico['mensajes'] / len(df)).round(2)
display(resumen_lexico)
resumen_lexico['mensajes'].plot.bar(rot=0, title='Resultado del método léxico (no son etiquetas reales)')
plt.ylabel('Mensajes')
plt.tight_layout()
plt.show()

## 5. Ampliación opcional: modelos preentrenados en español

Esta parte necesita Internet para descargar los pesos y espacio para almacenarlos. Una CPU sirve para los 30 mensajes seleccionados; una GPU es opcional. La primera ejecución puede tardar varios minutos. Si no hay acceso a Hugging Face, puedes completar las secciones anteriores sin activar esta parte.

En un entorno sin las bibliotecas, ejecuta en una celda:

```python
%pip install "transformers>=4.40,<5" torch
```

Después reinicia el kernel si procede, cambia `EJECUTAR_MODELOS = True` en la configuración y ejecuta desde el principio. Los modelos se conservan en la caché de Hugging Face. No enviamos los tweets a una API de inferencia: la inferencia se realiza localmente tras descargar los modelos.

Usamos [BETO sentimiento](https://huggingface.co/finiteautomata/beto-sentiment-analysis), un modelo para español con etiquetas `NEG`, `NEU`, `POS`. Los marcadores de URLs y usuarios son una normalización sencilla; no garantizan mejorar todas las predicciones. El valor `score` es la confianza asignada por el modelo a su clase elegida, **no una probabilidad calibrada de acertar**.

In [ ]:
if MAX_TEXTOS_MODELOS is not None and (not isinstance(MAX_TEXTOS_MODELOS, int) or MAX_TEXTOS_MODELOS < 1):
    raise ValueError('MAX_TEXTOS_MODELOS debe ser un entero positivo o None.')
n_modelos = len(df) if MAX_TEXTOS_MODELOS is None else min(MAX_TEXTOS_MODELOS, len(df))
df_modelos = df.sample(n=n_modelos, random_state=SEMILLA).sort_index().copy()
# Reiniciar resultados evita reutilizar una ejecución anterior al cambiar la configuración.
resultados_sentimiento = None
resultados_ner = None
entidades_df = None
if EJECUTAR_MODELOS:
    try:
        import torch
        from transformers import pipeline
    except ImportError as exc:
        raise ImportError('Instala torch y transformers como se indica arriba y reinicia el kernel.') from exc
    dispositivo = 0 if torch.cuda.is_available() else -1
    print(f'Mensajes: {len(df_modelos)}; dispositivo:', 'GPU' if dispositivo == 0 else 'CPU')
    modelo_sentimiento = 'finiteautomata/beto-sentiment-analysis'
    sa_pipeline = pipeline('text-classification', model=modelo_sentimiento, device=dispositivo)
    resultados_sentimiento = sa_pipeline(df_modelos['texto_modelo'].tolist(), batch_size=8,
                                         truncation=True, max_length=128)
    df_modelos['sentimiento_modelo'] = [r['label'] for r in resultados_sentimiento]
    df_modelos['confianza_modelo'] = [r['score'] for r in resultados_sentimiento]
    if not set(df_modelos['sentimiento_modelo']).issubset({'POS', 'NEU', 'NEG'}):
        raise ValueError('Las etiquetas del modelo han cambiado: revisa su configuración antes de interpretar.')
    display(df_modelos[['texto', 'sentimiento_lexico', 'sentimiento_modelo', 'confianza_modelo']].head(10))
else:
    print('Modelos desactivados: se ha completado el análisis léxico, no se han generado predicciones neuronales.')

In [ ]:
if resultados_sentimiento is not None:
    resumen_modelo = df_modelos['sentimiento_modelo'].value_counts().reindex(['POS', 'NEU', 'NEG'], fill_value=0).rename('mensajes').to_frame()
    resumen_modelo['porcentaje'] = (100 * resumen_modelo['mensajes'] / len(df_modelos)).round(2)
    print(f'Resultados SOLO de los {len(df_modelos)} mensajes procesados por el modelo:')
    display(resumen_modelo)
    display(pd.crosstab(df_modelos['sentimiento_lexico'], df_modelos['sentimiento_modelo']))
    print('Ejemplos de menor confianza (prioridad para revisión, no errores confirmados):')
    display(df_modelos.nsmallest(5, 'confianza_modelo')[['texto', 'sentimiento_modelo', 'confianza_modelo']])
else:
    print('Activa los modelos para comparar sus resultados. SIN_EVIDENCIA y NEU no son equivalentes.')

## 6. Ampliación opcional: entidades nombradas (NER)

Usamos [BETO ajustado para NER en español](https://huggingface.co/mrm8488/bert-spanish-cased-finetuned-ner), en vez de un modelo ajustado sobre textos en inglés. Esperamos personas (`PER`), lugares (`LOC`), organizaciones (`ORG`) y otras entidades (`MISC`).

El texto conserva las mayúsculas y los nombres. Agrupamos subpalabras y reconstruimos cada entidad con sus posiciones en el **texto original**. Un lugar mencionado no prueba dónde se publicó el mensaje ni dónde reside su autor. Hashtags, faltas ortográficas y lenguaje informal pueden causar errores. Tanto sentimiento como NER truncan a 128 tokens: un texto más largo puede quedar parcialmente sin analizar.

In [ ]:
if EJECUTAR_MODELOS:
    modelo_ner = 'mrm8488/bert-spanish-cased-finetuned-ner'
    ner_pipeline = pipeline('token-classification', model=modelo_ner,
                            aggregation_strategy='simple', device=dispositivo)
    # TokenClassificationPipeline trunca con el límite del tokenizer.
    ner_pipeline.tokenizer.model_max_length = 128
    resultados_ner = ner_pipeline(df_modelos['texto'].tolist(), batch_size=8)
    filas_entidades = []
    for (indice, fila), entidades in zip(df_modelos.iterrows(), resultados_ner):
        for entidad in entidades:
            inicio, fin = int(entidad['start']), int(entidad['end'])
            filas_entidades.append({'fila': indice, 'id': fila['id'],
                                    'entidad': fila['texto'][inicio:fin],
                                    'tipo': entidad['entity_group'],
                                    'confianza': float(entidad['score'])})
    entidades_df = pd.DataFrame(filas_entidades, columns=['fila', 'id', 'entidad', 'tipo', 'confianza'])
    display(entidades_df.head(15))
    for (indice, fila), entidades in list(zip(df_modelos.iterrows(), resultados_ner))[:3]:
        print('Texto:', fila['texto'])
        print('Entidades:', [(fila['texto'][int(e['start']):int(e['end'])], e['entity_group']) for e in entidades])
else:
    print('NER desactivado; no se han extraído entidades automáticamente.')

In [ ]:
if entidades_df is not None:
    if entidades_df.empty:
        print('El modelo no ha encontrado entidades en esta muestra.')
    else:
        entidades_df['entidad_normalizada'] = entidades_df['entidad'].str.casefold().str.strip()
        # Contamos mensajes distintos, no repeticiones del mismo nombre en un mensaje.
        frecuentes = (entidades_df.groupby(['entidad_normalizada', 'tipo'])['fila']
                      .nunique().sort_values(ascending=False).head(10).rename('mensajes').reset_index())
        display(frecuentes)
        print('Revisa variantes: “Torrevieja” y “#Torrevieja” pueden no quedar unificadas.')

## 7. Evaluar antes de sacar conclusiones

El fichero no contiene etiquetas humanas de sentimiento ni de entidades, por lo que **no podemos calcular la exactitud real** a partir de las predicciones. La coincidencia entre dos métodos tampoco demuestra que ambos acierten.

1. Selecciona al azar al menos 30 mensajes. Dos personas los anotan por separado como `POS`, `NEU`, `NEG` o `DUDOSO`, pensando en el tono global del mensaje (no necesariamente hacia Torrevieja).
2. Comparad desacuerdos: ironía, varios sentimientos, mensajes informativos y ausencia de contexto. Acordad una guía antes de evaluar.
3. Sobre las etiquetas acordadas, comparad el modelo con una referencia de clase mayoritaria. Calculad macro-F1 y una matriz de confusión; documentad los casos dudosos excluidos.
4. Para NER revisad límites y tipos: reconocer “Universidad de Alicante” exige acertar tanto el fragmento como su categoría.

La muestra está condicionada por su recogida, mezcla épocas y puede contener publicidad o retuits. No extrapoles sus porcentajes a la población ni interpretes una mención como opinión o ubicación.

In [ ]:
columnas_revision = ['id', 'texto', 'sentimiento_lexico']
revision = df.sample(n=min(30, len(df)), random_state=SEMILLA)[columnas_revision].copy()
revision['etiqueta_humana'] = ''
revision['observaciones'] = ''
# Para anotar sin sesgo, oculta inicialmente sentimiento_lexico y cualquier predicción.
display(revision[['id', 'texto', 'etiqueta_humana', 'observaciones']].head())
# Exportación opcional; elige una carpeta de trabajo fuera de los datos originales:
# revision.to_csv('revision_sentimiento.csv', index=False, encoding='utf-8')

## 8. Ejercicios

1. **Limpieza:** compara “me gusta” con “no me gusta”. Añade un ejemplo con emoji y otro con ironía. ¿Qué pierde cada representación?
2. **Léxico:** añade cinco términos tras revisar ejemplos. Explica dos errores que esa ampliación no resuelve. No ajustes el léxico con los mismos mensajes que usarás para evaluarlo.
3. **Sentimiento:** calcula total y porcentaje de `POS`, `NEU` y `NEG` en la muestra del modelo. Compáralos con el léxico, distinguiendo `SIN_EVIDENCIA` de `NEU`. Indica siempre el denominador.
4. **Vocabulario por clase:** representa las diez palabras más frecuentes para cada sentimiento; opcionalmente crea nubes de palabras. ¿Reflejan el tema, el sentimiento o ambos?
5. **Entidades:** obtiene los diez lugares más mencionados contando mensajes distintos. Revisa cinco extracciones erróneas y propone cómo agrupar variantes de un nombre.
6. **Evaluación:** anota ejemplos y describe qué etiquetas confunde el modelo. Presenta un resultado cuantitativo y tres errores comentados, sin publicar datos personales de los autores.

**Entrega:** una breve conclusión sobre lo aprendido y lo que estos datos y métodos no permiten afirmar.